In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os.path as path
import utils
import glob

In [2]:
nuts_df = gpd.read_file(path.join(utils.raw_data_dir, "NUTS_RG_01M_2024_4326.shp"))

In [3]:
nuts_df = nuts_df.to_crs(epsg=3035)
nuts_df["area_km2"] = round(nuts_df.geometry.area / 1e6, 2)

In [4]:
nuts_df

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,geometry,area_km2
0,AL011,3,AL,Dibër,Dibër,NaN,NaN,NaN,"POLYGON ((5179983.544 2144882.937, 5179843.043...",2470.03
1,AL012,3,AL,Durrës,Durrës,NaN,NaN,NaN,"POLYGON ((5139659.722 2104822.953, 5140602.597...",772.41
2,AL013,3,AL,Kukës,Kukës,NaN,NaN,NaN,"POLYGON ((5152332.168 2213667.282, 5156282.301...",2391.18
3,AL031,3,AL,Berat,Berat,NaN,NaN,NaN,"POLYGON ((5162428.402 2029789.314, 5162544.551...",1801.02
4,AL032,3,AL,Fier,Fier,NaN,NaN,NaN,"POLYGON ((5131123.546 2047723.145, 5131166.175...",1882.53
...,...,...,...,...,...,...,...,...,...,...
1793,RO,0,RO,România,România,NaN,NaN,NaN,"MULTIPOLYGON (((5550240.663 2933328.113, 55511...",238368.46
1794,NO,0,NO,Norge,Norge,NaN,NaN,NaN,"MULTIPOLYGON (((4770074.139 6396256.37, 477027...",384372.33
1795,PL,0,PL,Polska,Polska,NaN,NaN,NaN,"MULTIPOLYGON (((4852825.195 3556096.333, 48551...",311951.43
1796,PT,0,PT,Portugal,Portugal,NaN,NaN,NaN,"MULTIPOLYGON (((2828494.611 2296190.204, 28279...",91901.65


In [5]:
population_data = pd.read_csv(path.join(utils.raw_data_dir, "estat_demo_r_pjangrp3.tsv"))
population_data = population_data[population_data["sex"] == "T"]
population_data = population_data[population_data["age"] == "TOTAL"]

In [6]:
last_col = population_data.columns[-1]

population_data[last_col] = population_data[last_col].apply(lambda x: [e for e in x.split("\t") if e != ": " ])
population_data["NUTS_ID"] = population_data[last_col].apply(lambda x: x[0])
population_data["population"] = population_data[last_col].apply(lambda x: x[-1])
population_data = population_data.drop(columns=[last_col, "sex", "age", "freq", "unit"])

In [7]:
nuts_df = pd.merge(nuts_df, population_data, on="NUTS_ID", how="outer")

In [8]:
nuts3_df = nuts_df[nuts_df["LEVL_CODE"] == 3].drop(columns=["LEVL_CODE", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE"])

In [9]:
crop_profile_files = glob.glob(path.join(utils.intermediate_data_dir, "nuts3_crop_profile", "*.geojson"))
crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files])

/var/folders/1_/j9jx95wx4zv29v6wh_ztx5_r0000gp/T/ipykernel_3350/1508858051.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files])


In [10]:
def calc_crop_area(in_dict):
    in_dict = eval(in_dict)
    out_dict = {}
    for k, v in in_dict.items():
        if k == 0:
            # default value for non-cropland pixels is 0
            continue
        crop_name = utils.cropland_type_dict[k]
        # NOTE: this next step is important and deserves explanation
        # the raster size of the cropland dataset is _precisely_ 10x10m for all pixels, as defined by the CRS
        # thus, each pixel has an area of 100m^2. We get the total area in m^2 by multiplying the pixel value by 100
        # to get from m^2->km^2 we need to divide by 1.000*1.000, i.e. 1.000.000
        # in other words, we divide by 10.000 or 1e4
        area_km = round(v * 1e-4, 2)
        out_dict[crop_name] = area_km
    return out_dict

crop_profile_df["cropland_km2_by_type"] = crop_profile_df["crop_profile"].apply(calc_crop_area)
crop_profile_df["cropland_km2"] = crop_profile_df["cropland_km2_by_type"].apply(lambda x: round(sum(x.values()), 2))

In [11]:
nuts3_df = pd.merge(nuts3_df, crop_profile_df, on="NUTS_ID", how="outer")
nuts3_df["perc_agriculture"] = 100 * nuts3_df["cropland_km2"] / nuts3_df["area_km2"]

In [20]:
nuts3_drought_data = gpd.read_file(path.join(utils.intermediate_data_dir, "drought_days_nuts3.geojson"))

In [22]:
nuts3_df = pd.merge(nuts3_df, nuts3_drought_data, on="NUTS_ID", how="left")

In [23]:
nuts3_df.columns

Index(['NUTS_ID', 'CNTR_CODE_x', 'NUTS_NAME_x', 'NAME_LATN_x', 'geometry_x',
       'area_km2', 'population', 'cropland_km2_by_type', 'cropland_km2',
       'perc_agriculture', 'CNTR_CODE_y', 'NAME_LATN_y', 'NUTS_NAME_y',
       'median_warning_days', 'median_alert_days', 'median_drought_days',
       'geometry_y'],
      dtype='object')

In [ ]:
nuts3_df = nuts3_df[["NUTS_ID", "CNTR_CODE_x", "NUTS_NAME_x", "NAME_LATN_x", "geometry_x", "area_km2", 'population', 'cropland_km2_by_type', 'cropland_km2', 'perc_agriculture', 'median_warning_days', 'median_alert_days', 'median_drought_days']]
nuts3_df = nuts3_df.rename(columns={"CNTR_CODE_x": "CNTR_CODE", "NUTS_NAME_x": "NUTS_NAME", "NAME_LATN_x": "NAME_LATN", "geometry_x": "geometry",})

In [15]:
nuts3_df.set_index("NUTS_ID", drop=True, inplace=True)

In [16]:
nuts3_df

,CNTR_CODE,NUTS_NAME,NAME_LATN,geometry,area_km2,population,cropland_km2_by_type,cropland_km2,perc_agriculture
NUTS_ID,,,,,,,,,
AL011,AL,Dibër,Dibër,"POLYGON ((5179983.544 2144882.937, 5179843.043...",2470.03,104624,"{'Wheat': 1.96, 'Barley': 1.04, 'Maize': 10.11...",23.11,0.935616
AL012,AL,Durrës,Durrës,"POLYGON ((5139659.722 2104822.953, 5140602.597...",772.41,222999,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",144.52,18.710270
AL013,AL,Kukës,Kukës,"POLYGON ((5152332.168 2213667.282, 5156282.301...",2391.18,60207,"{'Wheat': 0.59, 'Barley': 0.1, 'Maize': 1.67, ...",8.18,0.342091
AL014,AL,Lezhë,Lezhë,"POLYGON ((5169833.39 2143939.504, 5169746.713 ...",1663.01,96384,"{'Wheat': 8.56, 'Barley': 1.56, 'Maize': 21.55...",111.43,6.700501
AL015,AL,Shkodër,Shkodër,"POLYGON ((5120936.933 2221189.677, 5120820.01 ...",3528.44,149496,"{'Wheat': 5.83, 'Barley': 1.44, 'Maize': 35.0,...",219.51,6.221163
...,...,...,...,...,...,...,...,...,...
XK003,XK,Pejë,Pejë,"POLYGON ((5166993.008 2250206.411, 5167914.189...",1659.08,NaN,"{'Wheat': 30.48, 'Barley': 3.56, 'Maize': 48.2...",153.46,9.249705
XK004,XK,Prizren,Prizren,"POLYGON ((5214323.166 2218102.589, 5215179.019...",1432.34,NaN,"{'Wheat': 20.66, 'Barley': 5.05, 'Maize': 10.5...",87.84,6.132622
XK005,XK,Ferizaj,Ferizaj,"POLYGON ((5247429.692 2219255.961, 5248446.877...",1022.69,NaN,"{'Wheat': 8.84, 'Barley': 0.33, 'Maize': 18.27...",95.86,9.373319


In [17]:
groups = nuts3_df.groupby("CNTR_CODE")
for ctr, df in groups:
    df.drop(columns=["geometry"]).to_excel(path.join(utils.out_data_dir, "nuts3_stats_by_country", f"{ctr}.xlsx"))

In [19]:
nuts3_df.to_file(path.join(utils.out_data_dir, "nuts3_stats.geojson"))